# 01 — Data Collection

Fetch all raw data via `nba_api` and cache to `data/raw/` as parquet files.

**Output:** `data/raw/playoff_games_{season}.parquet`, `data/raw/team_metrics_{season}.parquet`

**Note on rest days:** Rest days are computed from the playoff game dates themselves (no separate game log fetch needed).

**Run time:** ~5–10 min first run; subsequent runs read from cache instantly.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from tqdm import tqdm

sys.path.insert(0, str(Path().resolve().parent))
from src.data import fetch_playoff_games, fetch_team_estimated_metrics

In [ ]:
# CONFIG — modify here only
SEASONS = [
    "2014-15",
    "2015-16",
    "2016-17",
    "2017-18",
    "2018-19",
    "2020-21",
    "2021-22",
    "2022-23",
    "2023-24",
]
FORCE_REFETCH = False  # set True to ignore cache and re-download

## 1. Fetch Playoff Game Logs

In [ ]:
all_games = []
for season in tqdm(SEASONS, desc="Playoff games"):
    df = fetch_playoff_games(season, force=FORCE_REFETCH)
    df["season"] = season
    all_games.append(df)

games = pd.concat(all_games, ignore_index=True)
print(
    f"{len(games)} total playoff game rows across {games['season'].nunique()} seasons"
)
games.head()

## 2. Fetch Team Estimated Metrics (ORtg, DRtg, pace)

In [ ]:
all_metrics = []
for season in tqdm(SEASONS, desc="Team metrics"):
    df = fetch_team_estimated_metrics(season, force=FORCE_REFETCH)
    df["season"] = season
    all_metrics.append(df)

metrics = pd.concat(all_metrics, ignore_index=True)
print(f"{len(metrics)} team-season rows")
metrics.head()

## 3. Validation

In [ ]:
print("Playoff game rows per season (2 rows per game — one per team):")
print(games.groupby("season").size().to_string())
print()
print(f"Columns: {list(games.columns)}")
print()
print("Team metrics columns:")
print(metrics.columns.tolist())
print()
print("Missing values in games:")
print(games.isnull().sum()[games.isnull().sum() > 0])